In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

import sys; sys.path.insert(0, "../src")
from quad_fuzzy import make_data, fit_predictors, embed_2d, pick_corners, barycentric_triangle


for noise in (0.03, 0.5):
    print(f"noise={noise}:", {k: round(v, 3) for k, v in fit_predictors(make_data(noise=noise)).items()})

noise=0.03: {'a': 0.914, 'b': 0.903, 'c': 0.915, 'd': 0.967}
noise=0.5: {'a': 0.081, 'b': 0.031, 'c': 0.044, 'd': 0.131}


In [2]:
from sklearn.decomposition import PCA

df = make_data(noise=0.03)
X = df[["a", "b", "c", "d"]]

for k in (2, 3):
    pca = PCA(n_components=k).fit(X)
    kept = pca.explained_variance_ratio_.sum()
    print(f"{k}D projection keeps {kept:.1%} of variance")

# the actual 2D coordinates we'll draw triangles on later:
pca2 = PCA(n_components=2)
xy = pca2.fit_transform(X)      # shape (500, 2) — each 4D point squashed to a plane
print("\nfirst 5 projected points:\n", xy[:5])

2D projection keeps 70.8% of variance
3D projection keeps 99.8% of variance

first 5 projected points:
 [[ 0.56381585 -0.41479207]
 [-0.31821328  0.48357387]
 [ 0.05896536  0.6094032 ]
 [ 0.45265945  0.31530543]
 [-0.01800437 -0.42264001]]


In [3]:
## Data resembles  1 - (a+b+c)/3 + tiny noise ,, it looks like a 3 dimensional object located in a 4 dimensional space.
#fidelity depends on whether the flattened position still reflects the point's true location — and that's decided by how much information the flattening threw away.
# using 2d model for human explainability

In [4]:
import numpy as np
from scipy.spatial import Delaunay

def coverage_of_minimal_simplex(dim, n_points=2000, seed=0):
    """
    Fraction of a scattered cloud that lands INSIDE a single minimal simplex.
    A minimal simplex in `dim` dimensions has dim+1 corners.
    We pick those corners from the cloud itself (farthest-point spread) so the
    simplex is a fair 'best minimal container', not a random one.
    """
    rng = np.random.default_rng(seed)
    pts = rng.uniform(0, 1, size=(n_points, dim))   # scattered cloud in the unit cube

    # farthest-point sampling to pick dim+1 well-spread corners
    k = dim + 1
    chosen = [int(rng.integers(n_points))]
    dist = np.linalg.norm(pts - pts[chosen[0]], axis=1)
    for _ in range(k - 1):
        nxt = int(np.argmax(dist))
        chosen.append(nxt)
        dist = np.minimum(dist, np.linalg.norm(pts - pts[nxt], axis=1))

    corners = pts[chosen]
    tri = Delaunay(corners)                    # a single simplex (dim+1 corners)
    inside = tri.find_simplex(pts) >= 0        # True where a point is inside it
    return inside.mean()

print(f"{'dim':>4} | {'corners':>7} | {'coverage':>9}")
print("-" * 26)
for dim in range(2, 9):
    cov = coverage_of_minimal_simplex(dim)
    print(f"{dim:>4} | {dim+1:>7} | {cov:>8.1%}")

 dim | corners |  coverage
--------------------------
   2 |       3 |    42.9%
   3 |       4 |    12.8%
   4 |       5 |     4.2%
   5 |       6 |     0.4%
   6 |       7 |     0.5%
   7 |       8 |     0.4%
   8 |       9 |     0.4%


In [5]:
from quad_fuzzy import make_data, embed_2d
df = make_data()
pca, xy = embed_2d(df)
idx = pick_corners(xy, 4)      # your just-written function, defined in the cell above
print("corner indices:", idx)
print("corner coords:\n", xy[idx])
print("cloud x-range:", xy[:,0].min().round(2), xy[:,0].max().round(2))
print("cloud y-range:", xy[:,1].min().round(2), xy[:,1].max().round(2))

corner indices: [425, 108, 220, 234]
corner coords:
 [[-0.26148287  0.00574472]
 [ 0.82445614  0.09927323]
 [ 0.28496289 -0.61330746]
 [-0.39338591  0.72293062]]
cloud x-range: -0.8 0.83
cloud y-range: -0.65 0.72


In [6]:
A, B, C = np.array([0,0]), np.array([1,0]), np.array([0,1])
print(barycentric_triangle([0,0],     A, B, C))   # on corner A -> expect (1,0,0)
print(barycentric_triangle([1/3,1/3], A, B, C))   # centroid    -> expect (~.33,~.33,~.33)
print(barycentric_triangle([1,1],     A, B, C))   # OUTSIDE      -> expect a NEGATIVE weight

(1.0, 0.0, 0.0)
(0.3333333333333334, 0.3333333333333333, 0.3333333333333333)
(-1.0, 1.0, 1.0)
